# Importação de Bibliotecas

In [7]:
import pandas as pd
import numpy as np
import seaborn as sns

from ucimlrepo import fetch_ucirepo # Importação de Dados
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# 1) Preparação dos Dados

In [16]:
rt_iot2022 = fetch_ucirepo(id=942)

# Seperação das features com a coluna Target
x = rt_iot2022.data.features
y = rt_iot2022.data.targets

df_rt_iot2022 = pd.concat([x, y], axis=1)
target_column = y.columns[0]

print(f'Shape: {df_rt_iot2022.shape}')
print(f'Target: {target_column}\n')

KeyboardInterrupt: 

### 1.1) Remoção de Duplicadas

In [ ]:
df_rt_iot2022 = df_rt_iot2022.drop_duplicates().reset_index(drop=True)

print(f'Shape: {df_rt_iot2022.shape}')
print(f'Target: {target_column}\n')

Shape: (117922, 84)
Target: Attack_type



### 1.2) Remoção de Colunas Constantes

In [ ]:
constant_columns = [c for c in df_rt_iot2022.columns if df_rt_iot2022[c].nunique() <= 1]

df_rt_iot2022 = df_rt_iot2022.drop(columns=constant_columns, axis=1)

### 1.3) Remoção das Colunas Redundantes

In [ ]:
redundant_columns = ['bwd_bulk_packets', 'fwd_subflow_bytes', 'fwd_pkts_payload.tot', 'flow_iat.min', 'flow_iat.max', 'flow_iat.avg', 'flow_iat.tot', 'bwd_data_pkts_tot', 'bwd_header_size_min', 'fwd_bulk_rate', 'fwd_bulk_packets', 'active.min', 'active.avg', 'active.tot', 'idle.max', 'fwd_iat.avg', 'bwd_iat.min', 'bwd_pkts_payload.avg', 'bwd_pkts_payload.std', 'bwd_pkts_payload.max', 'fwd_iat.max', 'fwd_iat.min', 'fwd_iat.tot', 'bwd_init_window_size', 'bwd_iat.std', 'fwd_iat.std', 'bwd_PSH_flag_count', 'fwd_pkts_payload.min', 'bwd_iat.max', 'bwd_iat.tot', 'bwd_iat.avg', 'fwd_header_size_max', 'flow_ACK_flag_count', 'bwd_pkts_per_sec', 'payload_bytes_per_second', 'idle.tot', 'fwd_subflow_pkts']

df_rt_iot2022 = df_rt_iot2022.drop(columns=redundant_columns, axis=1)

print(f'Shape: {df_rt_iot2022.shape}')
print(f'Target: {target_column}\n')

Shape: (117922, 46)
Target: Attack_type



# 2) Treinamento do Modelo

In [ ]:
df_rt_iot2022_updated = pd.get_dummies(
    data=df_rt_iot2022,
    columns=['proto', 'service'],
    drop_first=True, dtype=int
)

df_rt_iot2022_updated[target_column] = LabelEncoder().fit_transform(df_rt_iot2022_updated[target_column])

In [ ]:
X = df_rt_iot2022_updated.drop(columns=[target_column], axis=1)
y = df_rt_iot2022_updated[target_column]

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

TypeError: '<' not supported between instances of 'LabelEncoder' and 'LabelEncoder'

In [ ]:
rf_model = RandomForestClassifier(n_estimators=150, random_state=42)

In [ ]:
rf_model.fit(X_train, y_train)

In [ ]:
y_test_pred = rf_model.predict(y_test)

In [ ]:
print('Relatório de Classificação')
classification_report(y_test, y_test_pred)